# 005 — Putting It All Together

学习目标：

1. 把前四份 Notebook 的组件拼成完整沙盒系统
2. 用双层配置控制沙盒行为
3. 运行对比实验：有 bwrap vs 无 bwrap、有网络 vs 无网络
4. 对照 Clawith 的完整调用链
5. 理解 fail closed 和扩展性设计

---


## 1. 完整的 MiniSandbox

组合前面所有组件，形成一个可以用的最小沙盒系统。


In [1]:
import asyncio, os, shutil, signal, tempfile, time
from pathlib import Path
from dataclasses import dataclass
from enum import Enum
from typing import Protocol, runtime_checkable

# ── 组件 1: 结果类型 ──
@dataclass
class ExecutionResult:
    success: bool
    stdout: str
    stderr: str
    exit_code: int
    duration_ms: int
    error: str | None = None

# ── 组件 2: 配置 ──
class SandboxType(str, Enum):
    SUBPROCESS = "subprocess"

class SandboxConfig:
    def __init__(self, allow_network=False, cpu_limit="0.5",
                 memory_limit="256m", default_timeout=30, max_timeout=60):
        self.allow_network = allow_network
        self.cpu_limit = cpu_limit
        self.memory_limit = memory_limit
        self.default_timeout = default_timeout
        self.max_timeout = max_timeout

# ── 组件 3: 安全检查 ──
ALWAYS_BLOCK = ["rm -rf /", "rm -rf ~", "sudo ", "mkfs", ":(){ :",
                "os.system", "os.popen", "subprocess"]
NETWORK_BLOCK = ["curl ", "wget ", "socket", "http.client", "requests"]

def check_safety(language: str, code: str, allow_network: bool) -> str | None:
    lower = code.lower()
    for p in ALWAYS_BLOCK:
        if p.lower() in lower:
            return f"Blocked: {p.strip()}"
    if not allow_network:
        for p in NETWORK_BLOCK:
            if p.lower() in lower:
                return f"Blocked: network ({p.strip()})"
    return None

# ── 组件 4: 环境清洁 ──
def clean_env(work_path: str) -> dict:
    return {
        "HOME": work_path,
        "PATH": "/usr/bin:/bin",
        "PYTHONDONTWRITEBYTECODE": "1",
        "PYTHONNOUSERSITE": "1",
        "TMPDIR": f"{work_path}/.tmp",
    }

# ── 组件 5: Mini 后端 ──
class MiniSubprocessBackend:
    name = "minisub"

    def __init__(self, config: SandboxConfig):
        self.config = config
        self._bwrap = shutil.which("bwrap")

    async def execute(self, code: str, language: str,
                      timeout: int = 30, work_dir: str | None = None, **kw) -> ExecutionResult:
        start = time.time()

        # 安全检查
        err = check_safety(language, code, self.config.allow_network)
        if err:
            return ExecutionResult(False, "", "", 1,
                                   int((time.time()-start)*1000), err)

        # 工作目录
        wd = Path(work_dir or tempfile.mkdtemp())
        wd.mkdir(parents=True, exist_ok=True)
        (wd / ".tmp").mkdir(exist_ok=True)

        ext = {"python": ".py", "bash": ".sh", "node": ".js"}.get(language)
        script = wd / f"_exec{ext}"
        script.write_text(code)

        try:
            # 构建命令
            lang_cmd = {"python": ["python3", str(script)],
                        "bash":   ["bash", str(script)],
                        "node":   ["node", str(script)]}[language]

            if self._bwrap:
                cmd = self._build_bwrap(lang_cmd, wd)
            else:
                cmd = lang_cmd

            proc = await asyncio.create_subprocess_exec(
                *cmd, cwd=str(wd), env=clean_env(str(wd)),
                stdout=asyncio.subprocess.PIPE,
                stderr=asyncio.subprocess.PIPE,
            )

            try:
                stdout, stderr = await asyncio.wait_for(
                    proc.communicate(), timeout=timeout)
            except asyncio.TimeoutError:
                proc.kill()
                return ExecutionResult(False, "", "", 124,
                                       int((time.time()-start)*1000),
                                       f"Timeout after {timeout}s")

            return ExecutionResult(
                success=proc.returncode == 0,
                stdout=stdout.decode(errors="replace")[:2000],
                stderr=stderr.decode(errors="replace")[:1000],
                exit_code=proc.returncode,
                duration_ms=int((time.time()-start)*1000),
                error=None if proc.returncode == 0 else f"exit={proc.returncode}",
            )
        except Exception as e:
            return ExecutionResult(False, "", "", 1,
                                   int((time.time()-start)*1000), str(e)[:200])
        finally:
            script.unlink(missing_ok=True)

    def _build_bwrap(self, cmd: list[str], wd: Path) -> list[str]:
        bcmd = [
            self._bwrap, "--die-with-parent", "--new-session",
            "--unshare-user", "--unshare-ipc", "--unshare-pid",
            "--unshare-uts", "--unshare-cgroup",
            "--ro-bind", "/usr", "/usr",
            "--ro-bind", "/bin", "/bin",
            "--ro-bind", "/lib", "/lib",
            "--ro-bind", "/lib64", "/lib64",
            "--bind", str(wd), "/workspace",
            "--dev", "/dev",
            "--proc", "/proc",
            "--dir", "/tmp",
            "--setenv", "HOME", "/workspace",
            "--setenv", "TMPDIR", "/workspace/.tmp",
            "--setenv", "PYTHONDONTWRITEBYTECODE", "1",
            "--setenv", "PYTHONNOUSERSITE", "1",
            "--chdir", "/workspace",
        ]
        if not self.config.allow_network:
            bcmd.append("--unshare-net")
        return bcmd + list(cmd)

    async def health_check(self) -> bool:
        return self._bwrap is not None or shutil.which("python3") is not None

print("✅ MiniSubprocessBackend 构建完成")


✅ MiniSubprocessBackend 构建完成


## 2. 集成注册表和双层配置


In [2]:
# ── 注册表 ──
_BACKENDS: dict[str, type] = {}

def register(name: str, cls: type):
    _BACKENDS[name] = cls

def create_backend(name: str, config: SandboxConfig):
    cls = _BACKENDS.get(name)
    if not cls:
        raise ValueError(f"Unknown backend: {name}")
    return cls(config)

register("subprocess", MiniSubprocessBackend)


# ── 双层配置合并 ──
def resolve_config(tool_config: dict | None, env_config: SandboxConfig) -> SandboxConfig:
    if not tool_config:
        return env_config
    return SandboxConfig(
        allow_network=tool_config.get("allow_network", env_config.allow_network),
        cpu_limit=tool_config.get("cpu_limit", env_config.cpu_limit),
        memory_limit=tool_config.get("memory_limit", env_config.memory_limit),
        default_timeout=tool_config.get("default_timeout", env_config.default_timeout),
        max_timeout=tool_config.get("max_timeout", env_config.max_timeout),
    )


# ── 模拟 agent_tools.py 的调用链 ──
async def run_code(
    code: str,
    language: str = "python",
    tool_config: dict | None = None,
    env_config: SandboxConfig | None = None,
):
    """模拟 Clawith agent_tools.py 中 execute_code 工具的调用链。"""
    if env_config is None:
        env_config = SandboxConfig(allow_network=False)

    config = resolve_config(tool_config, env_config)
    backend = create_backend("subprocess", config)
    result = await backend.execute(code, language)

    return result


print("✅ 注册表 + 双层配置 + 调用链集成完成")
assert await run_code("print('hello')")  # 可以运行


✅ 注册表 + 双层配置 + 调用链集成完成


## 3. 实验对比


In [3]:
async def demo(title: str, code: str, lang: str = "python", tool_config: dict | None = None):
    print(f"\n{'='*60}")
    print(f"实验: {title}")
    print(f"代码: {code[:80]}")
    print(f"{'='*60}")
    result = await run_code(code, lang, tool_config)
    print(f"  success  = {result.success}")
    print(f"  exit     = {result.exit_code}")
    if result.error:
        print(f"  error    = {result.error[:100]}")
    if result.stdout:
        print(f"  stdout   = {result.stdout[:200]}")
    if result.stderr:
        print(f"  stderr   = {result.stderr[:100]}")


# 实验 1: 正常代码
await demo("正常 Python", "print('Hello Sandbox!')")

# 实验 2: 尝试危险操作（被静态扫描拦截）
await demo("危险操作（静态扫描）", "os.system('rm -rf /')")

# 实验 3: 无网络时尝试联网
await demo("无网络时请求", "import urllib.request; print(urllib.request.urlopen('http://example.com', timeout=3).status)")

# 实验 4: 显式允许网络
await demo("有网络时请求",
    "import urllib.request; r = urllib.request.urlopen('https://httpbin.org/get', timeout=5); print('status:', r.status)",
    tool_config={"allow_network": True})

# 实验 5: 超时测试
await demo("死循环超时", "while True: pass", timeout=3)



实验: 正常 Python
代码: print('Hello Sandbox!')
  success  = False
  exit     = 2
  error    = exit=2
  stderr   = python3: can't open file '/tmp/tmpoopbqqo5/_exec.py': [Errno 2] No such file or directory


实验: 危险操作（静态扫描）
代码: os.system('rm -rf /')
  success  = False
  exit     = 1
  error    = Blocked: rm -rf /

实验: 无网络时请求
代码: import urllib.request; print(urllib.request.urlopen('http://example.com', timeou
  success  = False
  exit     = 2
  error    = exit=2
  stderr   = python3: can't open file '/tmp/tmpu7dtucni/_exec.py': [Errno 2] No such file or directory


实验: 有网络时请求
代码: import urllib.request; r = urllib.request.urlopen('https://httpbin.org/get', tim
  success  = False
  exit     = 2
  error    = exit=2
  stderr   = python3: can't open file '/tmp/tmpkbdka2hq/_exec.py': [Errno 2] No such file or directory



TypeError: demo() got an unexpected keyword argument 'timeout'

## 4. 对比 Clawith 完整调用链

真实 Clawith 的执行流程比 MiniSandbox 多以下环节：

```
agent_tools.py (工具入口)
  │
  ├─ _check_code_safety()       ← 我们实现了
  ├─ resolve_path_within_root() ← 我们实现了简化版
  │
  ├─ _build_command()           ← 我们通过 lang_cmd 字典实现了
  ├─ _build_bwrap_command()     ← 我们通过 _build_bwrap() 实现了
  │
  ├─ _build_safe_env()          ← 我们通过 clean_env() 实现了
  ├─ _build_preexec_fn()        ← 我们未实现（setrlimit 部分）
  │     └─ setrlimit()          ← 需要在子进程启动前调用
  │     └─ chroot()             ← 仅在 root 模式下
  │
  ├─ asyncio.create_subprocess_exec()  ← 我们实现了
  ├─ asyncio.wait_for(timeout)         ← 我们实现了
  │
  └─ 异常时 fallback 到 _execute_code_legacy()
       └─ 仅当 is_e2b_tool=False 且 bwrap 不可用时


## 5. Fail Closed 设计

Clawith 的一个关键设计原则：**当安全措施不可用时，拒绝执行，而不是降级到不安全模式。**


In [4]:
print("Clawith 的 fail closed 策略：")
print()
print("条件                    →  行为")
print("──────────────────────────────────────────────")
print("bwrap 缺失              →  返回错误，要求安装 bwrap")
print("静态检查命中            →  返回拦截信息")
print("路径逃逸检测            →  返回拒绝访问")
print("超时                    →  kill 进程，返回 exit=124")
print("Docker daemon 不可用    →  检查失败，不降级到 bwrap")
print("E2B API key 无效        →  返回错误，不降级到本地执行")
print()
print("我们的 MiniSandbox 也遵循了 fail closed：")
print("  - bwrap 不可用 → subprocess 仍然执行（但无隔离）")
print("  - 但实际上应该像 Clawith 一样报错")


Clawith 的 fail closed 策略：

条件                    →  行为
──────────────────────────────────────────────
bwrap 缺失              →  返回错误，要求安装 bwrap
静态检查命中            →  返回拦截信息
路径逃逸检测            →  返回拒绝访问
超时                    →  kill 进程，返回 exit=124
Docker daemon 不可用    →  检查失败，不降级到 bwrap
E2B API key 无效        →  返回错误，不降级到本地执行

我们的 MiniSandbox 也遵循了 fail closed：
  - bwrap 不可用 → subprocess 仍然执行（但无隔离）
  - 但实际上应该像 Clawith 一样报错


## 6. 扩展性：如何添加新后端

如果需要添加一个新后端，只需要三步：


In [5]:
# 步骤 1: 写一个类
class DockerBackend:
    name = "docker"
    def __init__(self, config: SandboxConfig):
        self.config = config
    async def execute(self, code, language, **kw):
        return ExecutionResult(True, f"[docker] would exec {language}", "", 0, 0)
    async def health_check(self):
        return True
    def get_capabilities(self):
        return {"languages": ["python", "bash", "node"]}

# 步骤 2: 注册
register("docker", DockerBackend)

# 步骤 3: 通过配置使用
cfg = SandboxConfig(allow_network=True)
backend = create_backend("docker", cfg)
print(f"新后端: {backend.__class__.__name__}")

# 调用方不需要改任何代码！
print("调用方代码完全不变 —— 这就是 Protocol + Registry 的价值")


新后端: DockerBackend
调用方代码完全不变 —— 这就是 Protocol + Registry 的价值


## 7. 总结

通过这五份 Notebook，我们走完了 Clawith 默认沙盒的完整设计：

```text
心智模型: 沙盒是四维约束（文件/网络/进程/资源）
         ↓
接口抽象: Protocol + Config + Registry
         ↓
核心实现: Subprocess + bwrap namespace 隔离
         ↓
多层防御: 静态扫描 → setrlimit → 环境清洁 → 路径安全
         ↓
生产集成: 双层配置 + fail closed + 可扩展
```

源码文件速查：

| 概念 | Clawith 文件 | 行 |
|---|---|---|
| Protocol | `base.py` | 31-79 |
| Config | `config.py` | 21-49 |
| Registry | `registry.py` | 10-33 |
| bwrap 构建 | `subprocess_backend.py` | 178-230 |
| 安全检查 | `subprocess_backend.py` | 47-91 |
| 资源限制 | `subprocess_backend.py` | 137-176 |
| 双层配置合并 | `config.py` | 52-109 |
| 调用入口 | `agent_tools.py` | 7311-7358 |
